In [ ]:
# ============================================
# W&B Lab1 - XGBoost with Weights & Biases
# Dataset: Breast Cancer (sklearn built-in)
# ============================================

import os
os.environ["WANDB__SERVICE_WAIT"] = "300"
os.environ["WANDB_MODE"] = "disabled"

import wandb
import numpy as np
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Step 1: Load and prepare data
data = load_breast_cancer()
X = data.data
y = data.target

train_X, test_X, train_Y, test_Y = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {train_X.shape}, Test: {test_X.shape}")

# Step 2: Initialize W&B run
run = wandb.init(
    project="Lab1-xgboost-tracking",
    name="xgboost_breast_cancer",
)

# Step 3: Setup XGBoost parameters and log config
param = {
    "objective": "binary:logistic",
    "eta": 0.1,
    "max_depth": 6,
    "nthread": 4,
    "eval_metric": "logloss",
}
wandb.config.update(param)
print(f"Config logged: {param}")

# Step 4: Create DMatrix and train
xg_train = xgb.DMatrix(train_X, label=train_Y)
xg_test = xgb.DMatrix(test_X, label=test_Y)

watchlist = [(xg_train, "train"), (xg_test, "test")]
num_round = 50

print("\nTraining XGBoost...")
bst = xgb.train(param, xg_train, num_round, watchlist, verbose_eval=10)

# Step 5: Evaluate
pred_probs = bst.predict(xg_test)
pred = (pred_probs > 0.5).astype(int)
acc = accuracy_score(test_Y, pred)
error_rate = np.sum(pred != test_Y) / test_Y.shape[0]

print(f"\nTest Accuracy: {acc:.4f}")
print(f"Test Error Rate: {error_rate:.4f}")

# Step 6: Log metrics to W&B
wandb.log({"accuracy": acc, "error_rate": error_rate})
run.summary["Accuracy"] = acc
run.summary["Error Rate"] = error_rate

# Step 7: Confusion Matrix
cm = confusion_matrix(test_Y, pred)
print(f"\nConfusion Matrix:\n{cm}")

# Finish
run.finish()
print("\nDone! W&B run complete.")
print("Note: Running in disabled mode due to Windows service issue.")
print("On Linux/Mac or Colab, remove WANDB_MODE=disabled to log to W&B dashboard.")